In [0]:
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS gold;

-- Dim fecha a partir de orders
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT DISTINCT
  CAST(date(order_date) AS DATE) AS date_value,
  CAST(date_format(order_date,'yyyyMMdd') AS INT) AS sk_date,
  year(order_date) AS year,
  month(order_date) AS month,
  day(order_date) AS day,
  weekofyear(order_date) AS week_of_year,
  date_format(order_date,'E') AS dow,
  date_format(order_date,'E') IN ('Sat','Sun') AS is_weekend
FROM silver.orders;

-- Dim customer (snapshot actual)
CREATE OR REPLACE TABLE gold.dim_customer AS
SELECT 
  customer_id AS nk_customer_id,
  customer_id AS sk_customer,
  concat_ws(' ', customer_fname, customer_lname) AS customer_name,
  customer_email, customer_city, customer_state, customer_zipcode
FROM silver.customers;

-- Dim product (denormalizada)
CREATE OR REPLACE TABLE gold.dim_product AS
SELECT 
  p.product_id AS nk_product_id,
  p.product_id AS sk_product,
  p.product_name, p.product_description, p.product_price,
  p.category_id, p.category_name, p.department_id, p.department_name
FROM silver.products p;

-- Fact línea de pedido
CREATE OR REPLACE TABLE gold.fact_order_item AS
SELECT 
  o.order_id AS nk_order_id,
  oi.order_item_id AS nk_order_item_id,
  dp.sk_product,
  dc.sk_customer,
  CAST(date_format(o.order_date,'yyyyMMdd') AS INT) AS sk_date,
  oi.order_item_quantity AS quantity,
  oi.order_item_product_price AS unit_price,
  oi.order_item_subtotal AS line_subtotal
FROM silver.order_items oi
JOIN silver.orders o     ON oi.order_item_order_id = o.order_id
LEFT JOIN gold.dim_product dp ON oi.order_item_product_id = dp.nk_product_id
LEFT JOIN gold.dim_customer dc ON o.order_customer_id = dc.nk_customer_id;